# Beta Instability for the Nasdaq-100

This variant varies its long and short allocations with a market regime filter, then evaluates consecutive historical windows. Set BETA_UNIVERSE_CSV to a dated Nasdaq-100 constituent CSV with a symbol column. The snapshot stays fixed during the backtest. Use an appropriately dated snapshot to avoid applying later constituent selection to earlier periods. Yahoo Finance supplies the OHLCV and adjusted closes.


In [ ]:
"""
BETA INSTABILITY STRATEGY - WALK-FORWARD ANALYSIS (ENHANCED)
=============================================================
Date Range: 2022-01-01 to 2025-10-24

ENHANCEMENTS APPLIED:
---------------------
✓ Regime Detection: Volatility + Trend filter on QQQ (T-1)
✓ Complete Financing: Margin interest + Short dividends + Borrow costs
✓ Explicit T-1/T Flow: Documented with regime snapshot
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import os
from _market_data import download_ohlcv
import warnings
from typing import Dict, Tuple, Optional, List
from scipy import stats
from dataclasses import dataclass
from itertools import product
import re

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')


# ============================================================================
# PRE-EXECUTION VALIDATION SYSTEM
# ============================================================================

@dataclass
class WFAConfig:
    """Walk-Forward Analysis Configuration with Regime Detection and Parameter Stability"""

    # ═══════════════════════════════════════════════════════════════════════
    # API Configuration
    # ═══════════════════════════════════════════════════════════════════════

    # ═══════════════════════════════════════════════════════════════════════
    # Backtest Period
    # ═══════════════════════════════════════════════════════════════════════
    BACKTEST_START_DATE: str = "2022-01-01"
    BACKTEST_END_DATE: str = "2025-10-24"

    # ═══════════════════════════════════════════════════════════════════════
    # Universe
    # ═══════════════════════════════════════════════════════════════════════
    UNIVERSE: str = "NASDAQ100"
    MARKET_BENCHMARK: str = "QQQ"
    BENCHMARK_TICKER: str = "QQQ"

    # ═══════════════════════════════════════════════════════════════════════
    # Capital & Leverage
    # ═══════════════════════════════════════════════════════════════════════
    INITIAL_CAPITAL: float = 150000.0
    LEVERAGE: float = 1.3
    COMPOUNDING_MODE: bool = False  # Research mode: fixed capital sizing

    # ═══════════════════════════════════════════════════════════════════════
    # Regime Detection
    # ═══════════════════════════════════════════════════════════════════════
    REGIME_ENABLE: bool = True
    REGIME_VOL_THRESHOLD: float = 0.25
    REGIME_VOL_WINDOW: int = 21
    REGIME_TREND_WINDOW: int = 63
    REGIME_TREND_THRESHOLD: float = 0.02

    # ═══════════════════════════════════════════════════════════════════════
    # ✓ NEW: Parameter Stability Controls
    # ═══════════════════════════════════════════════════════════════════════
    ENABLE_TURNOVER_PENALTY: bool = True
    TURNOVER_PENALTY_WEIGHT: float = 0.15  # Reduce Sharpe by 0.15 per 100% annual turnover

    ENABLE_POSITION_CAP: bool = True
    MAX_POSITION_PCT: float = 0.15  # Max 15% capital per position

    # ═══════════════════════════════════════════════════════════════════════
    # Transaction Costs
    # ═══════════════════════════════════════════════════════════════════════
    COMMISSION_PER_TRADE: float = 1.00
    SLIPPAGE_BPS: float = 5.0
    SLIPPAGE_BPS_REBALANCE: float = 10.0

    # ═══════════════════════════════════════════════════════════════════════
    # Financing Costs
    # ═══════════════════════════════════════════════════════════════════════
    MARGIN_INTEREST_RATE: float = 0.055
    SHORT_DIVIDEND_RATE: float = 0.015
    BORROW_COST_BPS: float = 30.0

    # ═══════════════════════════════════════════════════════════════════════
    # Data Quality
    # ═══════════════════════════════════════════════════════════════════════
    MIN_PRICE: float = 2.0
    MIN_VOLUME: float = 50000

    # ═══════════════════════════════════════════════════════════════════════
    # ✓ UPDATED: WFA Parameters (REDUCED GRID)
    # ═══════════════════════════════════════════════════════════════════════
    TRAIN_DAYS: int = 252
    VALIDATION_DAYS: int = 63
    TEST_DAYS: int = 63

    # Plotting configuration
    PLOT_FIGSIZE_WIDE: Tuple[int, int] = (16, 8)
    PLOT_FIGSIZE_STANDARD: Tuple[int, int] = (16, 6)
    PLOT_COLOR_TRAIN: str = '#1f77b4'
    PLOT_COLOR_VALIDATION: str = '#ff7f0e'
    PLOT_COLOR_TEST: str = '#2ca02c'
    PLOT_COLOR_STRATEGY: str = '#2E86AB'
    PLOT_COLOR_BENCHMARK: str = '#A23B72'
    PLOT_COLOR_LONG: str = '#06A77D'
    PLOT_COLOR_SHORT: str = '#D62246'
    PLOT_LINEWIDTH_MAIN: float = 2.5
    PLOT_LINEWIDTH_SECONDARY: float = 2.0
    PLOT_GRID_ALPHA: float = 0.3
    PLOT_DPI: int = 100

    def __post_init__(self):
        """
        Initialize optimization grid

        ✓ REDUCED FROM 243 TO 24 COMBINATIONS
        Strategy: Fix least important parameters, test only key drivers
        """

        # BEFORE: 3×3×3×3×3 = 243 combinations
        # AFTER:  2×2×2×3×1 = 24 combinations (90% reduction)

        self.OPTIMIZE_PARAMS = {
            # Beta calculation windows - KEEP LONGER WINDOWS (more stable)
            'BETA_WINDOW': [60, 90],          # Was [40, 60, 90] → reduced to 2
            'BETA_VOL_WINDOW': [60, 90],      # Was [40, 60, 90] → reduced to 2

            # Signal percentiles - TEST MODERATE VALUES
            'LONG_PERCENTILE': [0.10, 0.20],
            'SHORT_PERCENTILE': [0.80, 0.90],

            # Rebalance frequency - FIX TO QUARTERLY (reduce turnover)
            'REBALANCE_FREQUENCY': [42]       # Was [21, 42, 63] → FIXED to 63 (quarterly)
        }

        # Total combinations: 2×2×2×2×1 = 16
        # Even more conservative: Can reduce to 8 if needed

    @property
    def data_buffer_days(self) -> int:
        """Buffer days before backtest start for feature calculation"""
        max_window = 180
        return int(max_window * 1.5)

    @property
    def data_start_date(self) -> str:
        """Actual data fetch start (before backtest start)"""
        backtest_start = datetime.strptime(self.BACKTEST_START_DATE, "%Y-%m-%d")
        data_start = backtest_start - timedelta(days=self.data_buffer_days)
        return data_start.strftime("%Y-%m-%d")

    @property
    def slippage_rate(self) -> float:
        """Convert bps to decimal"""
        return self.SLIPPAGE_BPS / 10000.0

    @property
    def slippage_rate_rebalance(self) -> float:
        """Convert rebalance bps to decimal"""
        return self.SLIPPAGE_BPS_REBALANCE / 10000.0

    @property
    def borrow_cost_rate(self) -> float:
        """Convert bps to annual rate"""
        return self.BORROW_COST_BPS / 10000.0

    # ✓ AGREGAR ESTAS PROPIEDADES QUE FALTAN:
    @property
    def daily_borrow_cost(self) -> float:
        """Daily borrow cost rate"""
        return self.borrow_cost_rate / 252.0

    @property
    def daily_margin_rate(self) -> float:
        """Daily margin interest rate"""
        return self.MARGIN_INTEREST_RATE / 252.0

    @property
    def daily_short_dividend_rate(self) -> float:
        """Daily short dividend rate"""
        return self.SHORT_DIVIDEND_RATE / 252.0

    @property
    def total_param_combinations(self) -> int:
        total = 1
        for values in self.OPTIMIZE_PARAMS.values():
            total *= len(values)
        return total


# ============================================================================
# DATA FETCHER
# ============================================================================

class DataFetcher:
    """Fetches data from Yahoo Finance - No forward/backward filling"""

    def __init__(self):
        self.universe_path = os.environ.get("BETA_UNIVERSE_CSV")
        if not self.universe_path:
            raise ValueError("Set BETA_UNIVERSE_CSV to a dated constituent list with a symbol column.")

    def _read_snapshot(self) -> List[str]:
        frame = pd.read_csv(self.universe_path, dtype={"symbol": str})
        if "symbol" not in frame:
            raise ValueError("The member list needs a symbol column.")
        symbols = frame["symbol"].dropna().str.strip().str.upper().str.replace(".", "-", regex=False)
        tickers = list(dict.fromkeys(symbols[symbols.ne("")]))
        if not tickers:
            raise ValueError("The constituent list is empty.")
        return tickers

    def fetch_nasdaq100_tickers(self) -> List[str]:
        return self._read_snapshot()

    def fetch_historical_data(self, ticker: str, start_date: str, end_date: str) -> Optional[pd.DataFrame]:
        try:
            prices = download_ohlcv(ticker, start_date, end_date)
        except RuntimeError:
            return None
        if "Adj Close" not in prices or prices["Adj Close"].isna().mean() > 0.05:
            return None
        return prices.rename(columns={"Adj Close": "Adj_Close"}).rename_axis("Date")

    def fetch_universe_data(self, config: WFAConfig) -> Dict[str, pd.DataFrame]:
        print("\n" + "=" * 80)
        print("FETCHING UNIVERSE DATA")
        print("=" * 80)

        tickers = self.fetch_nasdaq100_tickers()
        all_tickers = list(set(tickers + [config.MARKET_BENCHMARK, config.BENCHMARK_TICKER]))

        print(f"Fetching {len(all_tickers)} tickers from {config.data_start_date} to {config.BACKTEST_END_DATE}...")

        data_dict = {}
        for i, ticker in enumerate(all_tickers):
            if (i + 1) % 10 == 0:
                print(f"  Progress: {i+1}/{len(all_tickers)}", end='\r')

            df = self.fetch_historical_data(ticker, config.data_start_date,
                                           config.BACKTEST_END_DATE)
            if df is not None:
                data_dict[ticker] = df

        print(f"\n✓ Successfully fetched: {len(data_dict)} tickers")

        if config.MARKET_BENCHMARK not in data_dict:
            raise ValueError(f"Missing benchmark: {config.MARKET_BENCHMARK}")

        return data_dict


# ============================================================================
# ✓ NEW: REGIME DETECTOR
# ============================================================================

class RegimeDetector:
    """Detect market regime using QQQ volatility and trend at T-1"""

    def __init__(self, config: WFAConfig):
        self.config = config

    def calculate_regime(self, benchmark_data: pd.DataFrame,
                        date: pd.Timestamp) -> Dict[str, any]:
        """
        Calculate regime at T-1 (current_date) for T execution.

        Returns dict with:
          - regime: 'high_vol', 'low_vol', 'uptrend', 'downtrend', 'neutral'
          - exposure_multiplier: 0.0 to 1.0
          - volatility: realized vol
          - trend_signal: +1, 0, -1
        """

        if not self.config.REGIME_ENABLE:
            return {
                'regime': 'neutral',
                'exposure_multiplier': 1.0,
                'volatility': 0.0,
                'trend_signal': 0
            }

        if date not in benchmark_data.index:
            return {
                'regime': 'unknown',
                'exposure_multiplier': 0.5,
                'volatility': np.nan,
                'trend_signal': 0
            }

        # Get historical data up to date (T-1)
        hist = benchmark_data.loc[:date]

        if len(hist) < max(self.config.REGIME_VOL_WINDOW,
                          self.config.REGIME_TREND_WINDOW):
            return {
                'regime': 'insufficient_data',
                'exposure_multiplier': 0.5,
                'volatility': np.nan,
                'trend_signal': 0
            }

        # Calculate volatility (annualized)
        returns = np.log(hist['Adj_Close'] / hist['Adj_Close'].shift(1))
        vol_window = returns.tail(self.config.REGIME_VOL_WINDOW)
        realized_vol = vol_window.std() * np.sqrt(252)

        # Calculate trend (price vs MA)
        trend_ma = hist['Adj_Close'].rolling(self.config.REGIME_TREND_WINDOW).mean()
        current_price = hist['Adj_Close'].iloc[-1]
        ma_value = trend_ma.iloc[-1]

        if pd.isna(ma_value) or pd.isna(realized_vol):
            return {
                'regime': 'insufficient_data',
                'exposure_multiplier': 0.5,
                'volatility': np.nan,
                'trend_signal': 0
            }

        # Determine regime
        high_vol = realized_vol > self.config.REGIME_VOL_THRESHOLD
        uptrend = current_price > ma_value * 1.02  # 2% above MA
        downtrend = current_price < ma_value * 0.98  # 2% below MA

        # Set exposure multiplier
        if high_vol and downtrend:
            regime = 'high_vol_down'
            exposure_mult = 0.25  # Reduce exposure significantly
        elif high_vol:
            regime = 'high_vol'
            exposure_mult = 0.5   # Moderate reduction
        elif downtrend:
            regime = 'downtrend'
            exposure_mult = 0.75  # Slight reduction
        elif uptrend:
            regime = 'uptrend'
            exposure_mult = 1.0   # Full exposure
        else:
            regime = 'neutral'
            exposure_mult = 0.85  # Slightly cautious

        trend_signal = 1 if uptrend else (-1 if downtrend else 0)

        return {
            'regime': regime,
            'exposure_multiplier': exposure_mult,
            'volatility': realized_vol,
            'trend_signal': trend_signal
        }


# ============================================================================
# BETA CALCULATOR
# ============================================================================

class BetaCalculator:
    """Calculate rolling betas and beta volatility"""

    def __init__(self, config: WFAConfig):
        self.config = config

    def calculate_returns(self, df: pd.DataFrame) -> pd.Series:
        """
        Calculate log returns from adjusted close prices
        ENHANCED: With fallback and error handling
        """

        if 'Adj_Close' not in df.columns:
            if 'Close' in df.columns:
                return np.log(df['Close'] / df['Close'].shift(1)).dropna()
            else:
                return pd.Series(dtype=float)

        returns = np.log(df['Adj_Close'] / df['Adj_Close'].shift(1)).dropna()
        return returns

    def calculate_rolling_beta_ols(self, stock_returns: pd.Series,
                                   market_returns: pd.Series,
                                   window: int) -> pd.Series:
        """Calculate rolling beta using OLS regression"""

        betas = pd.Series(index=stock_returns.index, dtype=float)

        for i in range(window, len(stock_returns)):
            y = stock_returns.iloc[i-window:i].values
            x = market_returns.iloc[i-window:i].values

            if len(x) < window or len(y) < window:
                continue

            # Remove any NaN values
            mask = ~(np.isnan(x) | np.isnan(y))
            x_clean = x[mask]
            y_clean = y[mask]

            if len(x_clean) < window * 0.8:  # Need at least 80% of data
                continue

            # OLS: beta = cov(x,y) / var(x)
            if np.var(x_clean) > 1e-10:  # Avoid division by zero
                beta = np.cov(x_clean, y_clean)[0, 1] / np.var(x_clean)
                betas.iloc[i] = beta

        return betas.dropna()

    def calculate_beta_volatility(self, betas: pd.Series, window: int) -> pd.Series:
        """Calculate rolling volatility of betas"""

        if len(betas) < window:
            return pd.Series(dtype=float)

        beta_vol = betas.rolling(window=window).std()
        return beta_vol.dropna()

    def calculate_all_betas(self, data_dict: Dict[str, pd.DataFrame],
                           beta_window: int = 60,
                           beta_vol_window: int = 40) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Calculate betas and beta volatilities for all stocks
        ENHANCED: With detailed debugging
        """

        print(f"\n  🔍 calculate_all_betas called:")
        print(f"     beta_window={beta_window}, beta_vol_window={beta_vol_window}")
        print(f"     data_dict has {len(data_dict)} tickers")

        if self.config.MARKET_BENCHMARK not in data_dict:
            print(f"  ❌ ERROR: Benchmark {self.config.MARKET_BENCHMARK} not in data_dict")
            return pd.DataFrame(), pd.DataFrame()

        market_returns = self.calculate_returns(data_dict[self.config.MARKET_BENCHMARK])

        print(f"     Market returns shape: {market_returns.shape}")
        if len(market_returns) > 0:
            print(f"     Market returns range: {market_returns.index[0]} to {market_returns.index[-1]}")

        if len(market_returns) < beta_window:
            print(f"  ❌ ERROR: Not enough market data. Have {len(market_returns)}, need {beta_window}")
            return pd.DataFrame(), pd.DataFrame()

        all_betas = {}
        all_beta_vols = {}
        successful_tickers = 0
        failed_tickers = 0

        for ticker in data_dict.keys():
            if ticker == self.config.MARKET_BENCHMARK:
                continue

            try:
                stock_returns = self.calculate_returns(data_dict[ticker])

                if len(stock_returns) == 0:
                    failed_tickers += 1
                    continue

                # Align dates
                common_dates = market_returns.index.intersection(stock_returns.index)

                if len(common_dates) < beta_window:
                    failed_tickers += 1
                    continue

                stock_ret_aligned = stock_returns.loc[common_dates]
                market_ret_aligned = market_returns.loc[common_dates]

                # Calculate rolling beta
                betas = self.calculate_rolling_beta_ols(
                    stock_ret_aligned, market_ret_aligned, beta_window
                )

                if len(betas) == 0:
                    failed_tickers += 1
                    continue

                # Calculate beta volatility
                beta_vols = self.calculate_beta_volatility(betas, beta_vol_window)

                if len(betas) > 0:
                    all_betas[ticker] = betas
                    successful_tickers += 1

                if len(beta_vols) > 0:
                    all_beta_vols[ticker] = beta_vols

            except Exception as e:
                failed_tickers += 1
                continue

        print(f"     Successfully calculated: {successful_tickers} tickers")
        print(f"     Failed: {failed_tickers} tickers")

        if len(all_betas) == 0:
            print(f"  ❌ ERROR: No successful beta calculations")
            return pd.DataFrame(), pd.DataFrame()

        betas_df = pd.DataFrame(all_betas)
        beta_vols_df = pd.DataFrame(all_beta_vols)

        print(f"     Final betas_df shape: {betas_df.shape}")
        print(f"     Final beta_vols_df shape: {beta_vols_df.shape}")

        if len(beta_vols_df) > 0:
            print(f"     beta_vols_df date range: {beta_vols_df.index[0]} to {beta_vols_df.index[-1]}")

        return beta_vols_df, betas_df



# ============================================================================
# SIGNAL GENERATOR
# ============================================================================

class SignalGenerator:
    """Generate long/short signals based on beta volatility"""

    def __init__(self, config: WFAConfig):
        self.config = config

    def apply_filters(self, data_dict: Dict[str, pd.DataFrame],
                     date: pd.Timestamp) -> List[str]:
        filtered = []

        for ticker, df in data_dict.items():
            if ticker in [self.config.MARKET_BENCHMARK, self.config.BENCHMARK_TICKER]:
                continue

            if date not in df.index:
                continue

            row = df.loc[date]

            if (pd.isna(row['Close']) or pd.isna(row['Volume']) or
                row['Close'] < self.config.MIN_PRICE or
                row['Volume'] < self.config.MIN_VOLUME):
                continue

            filtered.append(ticker)

        return filtered

    def generate_signals(self, beta_vols_df: pd.DataFrame,
                        data_dict: Dict[str, pd.DataFrame],
                        date: pd.Timestamp,
                        long_pct: float,
                        short_pct: float) -> Tuple[List[str], List[str]]:

        if date not in beta_vols_df.index:
            return [], []

        beta_vols = beta_vols_df.loc[date]
        valid_tickers = self.apply_filters(data_dict, date)

        available_tickers = [t for t in valid_tickers if t in beta_vols.index]
        beta_vols = beta_vols[available_tickers].dropna()

        if len(beta_vols) < 20:
            return [], []

        long_threshold = beta_vols.quantile(long_pct)
        short_threshold = beta_vols.quantile(short_pct)

        long_tickers = beta_vols[beta_vols <= long_threshold].index.tolist()
        short_tickers = beta_vols[beta_vols >= short_threshold].index.tolist()

        return long_tickers, short_tickers


# ============================================================================
# ✓ ENHANCED: BACKTEST ENGINE WITH REGIME & FULL FINANCING
# ============================================================================

class BacktestEngine:
    """Execute backtest with regime filter and complete financing costs"""

    def __init__(self, config: WFAConfig):
        self.config = config
        self.trades = []
        self.rebalance_dates = []
        self.last_positions = {}
        self.regime_detector = RegimeDetector(config)
        self.regime_history = []
        self.last_rebalance_frequency = None

    def calculate_transaction_cost(self, trade_value: float,
                                  is_rebalance: bool = False) -> float:
        """✓ ENHANCED: Higher slippage on rebalance days"""
        commission = self.config.COMMISSION_PER_TRADE

        if is_rebalance:
            slippage = abs(trade_value) * self.config.slippage_rate_rebalance
        else:
            slippage = abs(trade_value) * self.config.slippage_rate

        return commission + slippage

    def calculate_financing_costs(self, long_value: float, short_value: float,
                                cash_reserve: float) -> Dict:
        """
        ✓ NEW: Complete financing cost model

        Returns daily costs for:
        - borrow_cost: Fee to borrow shares for shorting
        - margin_interest: Interest on negative cash balance
        - short_dividends: Dividend payments on short positions
        - total_cost: Sum of all costs
        """

        # Stock borrow cost (on short exposure)
        borrow_cost = abs(short_value) * self.config.daily_borrow_cost  # ✓ CORRECTO

        # Margin interest (only if cash is negative)
        if cash_reserve < 0:
            margin_interest = abs(cash_reserve) * self.config.daily_margin_rate
        else:
            margin_interest = 0.0

        # Short dividend payments (estimate)
        short_dividends = abs(short_value) * self.config.daily_short_dividend_rate

        total_cost = borrow_cost + margin_interest + short_dividends

        return {
            'borrow_cost': borrow_cost,
            'margin_interest': margin_interest,
            'short_dividends': short_dividends,
            'total_financing': total_cost
        }


    def calculate_position_sizes(self, long_tickers: List[str],
                                short_tickers: List[str],
                                sizing_capital: float,
                                regime_multiplier: float = 1.0) -> Tuple[Dict, Dict]:
        """
        ✓ ENHANCED: Apply regime multiplier to exposure

        T-1 SNAPSHOT:
          - sizing_capital: from Close[T-1] NAV
          - regime_multiplier: from T-1 regime detection
          - Result stored implicitly for T execution
        """

        # Apply regime adjustment
        adjusted_leverage = self.config.LEVERAGE * regime_multiplier

        long_exposure = sizing_capital * adjusted_leverage / 2
        short_exposure = sizing_capital * adjusted_leverage / 2

        if len(long_tickers) > 0:
            long_weight = long_exposure / len(long_tickers)
            long_weights = {ticker: long_weight for ticker in long_tickers}
        else:
            long_weights = {}

        if len(short_tickers) > 0:
            short_weight = short_exposure / len(short_tickers)
            short_weights = {ticker: short_weight for ticker in short_tickers}
        else:
            short_weights = {}

        return long_weights, short_weights

    def execute_rebalance(self, current_positions: Dict,
                         target_long: Dict, target_short: Dict,
                         prices: Dict, date: pd.Timestamp,
                         cash_reserve: float) -> Tuple[Dict, float, List]:

        self.rebalance_dates.append(date)

        new_positions = {}
        trade_records = []
        total_costs = 0.0

        all_tickers = set(list(current_positions.keys()) +
                         list(target_long.keys()) +
                         list(target_short.keys()))

        for ticker in all_tickers:
            if ticker not in prices or pd.isna(prices[ticker]):
                continue

            price = prices[ticker]
            current_shares = current_positions.get(ticker, {}).get('shares', 0.0)

            if ticker in target_long:
                target_value = target_long[ticker]
                target_side = 'long'
            elif ticker in target_short:
                target_value = -target_short[ticker]
                target_side = 'short'
            else:
                target_value = 0.0
                target_side = None

            target_shares = target_value / price if target_value != 0 else 0.0
            shares_to_trade = target_shares - current_shares

            if abs(shares_to_trade) > 0.001:
                trade_value = shares_to_trade * price
                costs = self.calculate_transaction_cost(trade_value, is_rebalance=True)
                total_costs += costs

                trade_records.append({
                    'Date': date, 'Ticker': ticker, 'Side': target_side or 'close',
                    'Shares': shares_to_trade, 'Price': price,
                    'Value': trade_value, 'Commission': costs
                })

                if abs(target_shares) > 0.001:
                    new_positions[ticker] = {
                        'shares': target_shares,
                        'side': target_side,
                        'entry_price': price
                    }
            else:
                if abs(current_shares) > 0.001:
                    new_positions[ticker] = current_positions[ticker]

        self.last_positions = new_positions.copy()
        cash_reserve_after_costs = cash_reserve - total_costs
        return new_positions, cash_reserve_after_costs, trade_records

    def run_backtest(self,
                 data_dict: Dict[str, pd.DataFrame],
                 beta_vols_df: pd.DataFrame,
                 signal_generator: SignalGenerator,
                 start_date: pd.Timestamp,
                 end_date: pd.Timestamp,
                 long_pct: float,
                 short_pct: float,
                 rebal_freq: int) -> pd.DataFrame:
        """
        ENHANCED: Full T-1/T flow with regime detection + AUDIT LOGGING

        DAY T-1 (current_date):
        1. Value portfolio at Close[T-1]
        2. Detect regime using QQQ data up to T-1
        3. Generate signals from features at T-1
        4. Calculate sizing: capital = INITIAL_CAPITAL (or current NAV)
        5. Apply regime multiplier to leverage
        6. Store sizing snapshot (implicit)

        DAY T (next_date):
        1. Execute trades at Open[T] using T-1 sizing
        2. Apply transaction costs
        3. Value portfolio at Close[T]
        4. Apply daily financing costs
        5. No look-ahead: sizing does not use T data
        """

        # ═══════════════════════════════════════════════════════════════════════
        # ✓ NEW: Audit configuration
        # ═══════════════════════════════════════════════════════════════════════
        ENABLE_AUDIT_LOG = True  # Set to False to disable
        AUDIT_THRESHOLD_RETURN = 0.05  # Flag daily returns > 5%
        AUDIT_THRESHOLD_POSITION = 0.20  # Flag positions > 20% of capital

        audit_log = []

        def log_audit(message: str, level: str = "INFO"):
            """Helper for detailed logging"""
            if ENABLE_AUDIT_LOG:
                audit_log.append({
                    'timestamp': datetime.now(),
                    'level': level,
                    'message': message
                })
                if level in ["WARNING", "ERROR"]:
                    print(f"  ⚠️  {message}")

        # ═══════════════════════════════════════════════════════════════════════
        # Initialization
        # ═══════════════════════════════════════════════════════════════════════

        self.trades = []
        self.rebalance_dates = []
        self.regime_history = []

        market_data = data_dict[self.config.MARKET_BENCHMARK]
        all_dates = market_data.loc[
            (market_data.index >= start_date) & (market_data.index <= end_date)
        ].index

        if len(all_dates) < 2:
            log_audit("Insufficient dates for backtest", "ERROR")
            return pd.DataFrame()

        log_audit(f"Backtest period: {all_dates[0]} to {all_dates[-1]}")
        log_audit(f"Rebalance frequency: {rebal_freq} days")

        results = []
        current_positions = {}
        cash_reserve = self.config.INITIAL_CAPITAL
        days_since_rebalance = 999

        # ═══════════════════════════════════════════════════════════════════════
        # Main Loop
        # ═══════════════════════════════════════════════════════════════════════

        for idx in range(len(all_dates) - 1):
            current_date = all_dates[idx]      # T-1
            next_date = all_dates[idx + 1]     # T

            # ───────────────────────────────────────────────────────────────────
            # DAY T-1: REBALANCE DECISION
            # ───────────────────────────────────────────────────────────────────

            if days_since_rebalance >= rebal_freq:
                # Step 1: Detect regime at T-1
                regime_info = self.regime_detector.calculate_regime(
                    market_data, current_date
                )

                self.regime_history.append({
                    'Date': current_date,
                    **regime_info
                })

                log_audit(
                    f"T-1 Regime: {regime_info['regime']}, "
                    f"Multiplier: {regime_info['exposure_multiplier']:.2f}, "
                    f"Vol: {regime_info['volatility']:.2%}"
                )

                # Step 2: Generate signals at T-1
                long_tickers, short_tickers = signal_generator.generate_signals(
                    beta_vols_df, data_dict, current_date, long_pct, short_pct
                )

                if len(long_tickers) == 0 and len(short_tickers) == 0:
                    target_long = {}
                    target_short = {}
                    log_audit("No signals generated - skipping rebalance", "WARNING")
                else:
                    # Step 3: Calculate NAV at Close[T-1]
                    close_prices_t1 = {}
                    for ticker in current_positions.keys():
                        if ticker in data_dict and current_date in data_dict[ticker].index:
                            close_prices_t1[ticker] = data_dict[ticker].loc[current_date, 'Adj_Close']

                    long_value = sum(
                        pos['shares'] * close_prices_t1.get(ticker, 0)
                        for ticker, pos in current_positions.items()
                        if pos['side'] == 'long' and ticker in close_prices_t1
                    )

                    short_value = sum(
                        -pos['shares'] * close_prices_t1.get(ticker, 0)
                        for ticker, pos in current_positions.items()
                        if pos['side'] == 'short' and ticker in close_prices_t1
                    )

                    current_nav = cash_reserve + long_value + short_value

                    # ✓ AUDIT: Verify reasonable NAV
                    if current_nav < self.config.INITIAL_CAPITAL * 0.5:
                        log_audit(
                            f"WARNING: NAV dropped to ${current_nav:,.2f} "
                            f"({current_nav/self.config.INITIAL_CAPITAL:.1%} of initial)",
                            "WARNING"
                        )

                    # Step 4: Fix sizing capital = INITIAL_CAPITAL for research
                    if self.config.COMPOUNDING_MODE:
                        sizing_capital = current_nav
                        log_audit(f"Compounding mode: sizing on NAV ${sizing_capital:,.2f}")
                    else:
                        sizing_capital = self.config.INITIAL_CAPITAL

                    # Step 5: Calculate position sizes with regime adjustment
                    target_long, target_short = self.calculate_position_sizes(
                        long_tickers, short_tickers, sizing_capital,
                        regime_multiplier=regime_info['exposure_multiplier']
                    )

                    # ✓ NEW: Maximum position cap
                    MAX_POSITION_PCT = AUDIT_THRESHOLD_POSITION
                    max_position_value = self.config.INITIAL_CAPITAL * MAX_POSITION_PCT

                    for ticker in list(target_long.keys()):
                        original_value = target_long[ticker]
                        if abs(original_value) > max_position_value:
                            target_long[ticker] = np.sign(original_value) * max_position_value
                            log_audit(
                                f"CAPPED {ticker} long: ${original_value:,.0f} → ${max_position_value:,.0f}",
                                "WARNING"
                            )

                    for ticker in list(target_short.keys()):
                        original_value = target_short[ticker]
                        if abs(original_value) > max_position_value:
                            target_short[ticker] = np.sign(original_value) * max_position_value
                            log_audit(
                                f"CAPPED {ticker} short: ${original_value:,.0f} → ${max_position_value:,.0f}",
                                "WARNING"
                            )

                # ─────────────────────────────────────────────────────────────
                # DAY T: EXECUTE AT OPEN
                # ─────────────────────────────────────────────────────────────

                execution_date = next_date
                open_prices = {}
                for ticker in set(list(current_positions.keys()) +
                                list(target_long.keys()) +
                                list(target_short.keys())):
                    if ticker in data_dict and execution_date in data_dict[ticker].index:
                        open_prices[ticker] = data_dict[ticker].loc[execution_date, 'Open']

                # ✓ AUDIT: Check for extreme gaps
                for ticker, open_price in open_prices.items():
                    if ticker in close_prices_t1:
                        gap = (open_price / close_prices_t1[ticker]) - 1
                        if abs(gap) > 0.10:  # Gap > 10%
                            log_audit(
                                f"EXTREME GAP in {ticker}: {gap:.1%} "
                                f"(Close[T-1]: ${close_prices_t1[ticker]:.2f}, Open[T]: ${open_price:.2f})",
                                "WARNING"
                            )

                current_positions, cash_reserve, trade_records = self.execute_rebalance(
                    current_positions, target_long, target_short, open_prices,
                    execution_date, cash_reserve
                )

                self.trades.extend(trade_records)
                self.rebalance_dates.append(execution_date)
                days_since_rebalance = 0
                rebalanced = True
                valuation_date = next_date

                # ✓ AUDIT: Log rebalance summary
                total_trade_value = sum(abs(t['Value']) for t in trade_records)
                log_audit(
                    f"REBALANCE at {execution_date.date()}: "
                    f"{len(trade_records)} trades, ${total_trade_value:,.0f} volume"
                )
            else:
                valuation_date = next_date
                rebalanced = False
                days_since_rebalance += 1

            # ───────────────────────────────────────────────────────────────────
            # VALUATION AT CLOSE[T]
            # ───────────────────────────────────────────────────────────────────

            close_prices = {}
            for ticker in current_positions.keys():
                if ticker in data_dict and valuation_date in data_dict[ticker].index:
                    close_prices[ticker] = data_dict[ticker].loc[valuation_date, 'Adj_Close']

            long_value = sum(
                pos['shares'] * close_prices.get(ticker, 0)
                for ticker, pos in current_positions.items()
                if pos['side'] == 'long' and ticker in close_prices
            )

            short_value = sum(
                -pos['shares'] * close_prices.get(ticker, 0)
                for ticker, pos in current_positions.items()
                if pos['side'] == 'short' and ticker in close_prices
            )

            # Calculate financing costs
            financing = self.calculate_financing_costs(long_value, short_value, cash_reserve)
            cash_reserve -= financing['total_financing']

            portfolio_value = cash_reserve + long_value + short_value

            # ✓ NEW: Audit extreme returns
            if len(results) > 0:
                prev_value = results[-1]['Portfolio_Value']
                daily_return = (portfolio_value / prev_value) - 1

                if abs(daily_return) > AUDIT_THRESHOLD_RETURN:
                    log_audit(
                        f"EXTREME RETURN at {valuation_date.date()}: {daily_return:.2%}\n"
                        f"  Portfolio: ${prev_value:,.0f} → ${portfolio_value:,.0f}\n"
                        f"  Long: ${long_value:,.0f}, Short: ${short_value:,.0f}, Cash: ${cash_reserve:,.0f}\n"
                        f"  Positions: {len([p for p in current_positions.values() if p['side']=='long'])} long, "
                        f"{len([p for p in current_positions.values() if p['side']=='short'])} short",
                        "WARNING"
                    )

                    # ✓ Detail positions on extreme days
                    if ENABLE_AUDIT_LOG:
                        print(f"\n  📊 Position Detail for {valuation_date.date()}:")
                        for ticker, pos in sorted(current_positions.items(),
                                                key=lambda x: abs(x[1]['shares'] * close_prices.get(x[0], 0)),
                                                reverse=True)[:10]:
                            if ticker in close_prices:
                                pos_value = pos['shares'] * close_prices[ticker]
                                pct_of_capital = abs(pos_value) / self.config.INITIAL_CAPITAL
                                print(f"     {ticker}: ${pos_value:,.0f} ({pct_of_capital:.1%} of capital)")

            results.append({
                'Date': valuation_date,
                'Portfolio_Value': portfolio_value,
                'Long_Value': long_value,
                'Short_Value': short_value,
                'Cash_Reserve': cash_reserve,
                'Num_Long': sum(1 for p in current_positions.values() if p['side'] == 'long'),
                'Num_Short': sum(1 for p in current_positions.values() if p['side'] == 'short'),
                'Rebalanced': rebalanced,
                'Borrow_Cost': financing['borrow_cost'],
                'Margin_Interest': financing['margin_interest'],
                'Short_Dividends': financing['short_dividends'],
                'Total_Financing': financing['total_financing'],
                'Days_Since_Rebalance': days_since_rebalance
            })

        # ═══════════════════════════════════════════════════════════════════════
        # Finalization and audit report
        # ═══════════════════════════════════════════════════════════════════════

        if len(results) == 0:
            log_audit("No results generated", "ERROR")
            return pd.DataFrame()

        results_df = pd.DataFrame(results).set_index('Date')
        results_df['Daily_Return'] = results_df['Portfolio_Value'].pct_change()
        results_df['Cumulative_Return'] = (1 + results_df['Daily_Return']).cumprod() - 1

        # ✓ PRINT AUDIT SUMMARY
        if ENABLE_AUDIT_LOG:
            warnings_count = sum(1 for log in audit_log if log['level'] == 'WARNING')
            errors_count = sum(1 for log in audit_log if log['level'] == 'ERROR')

            print(f"\n  📋 Audit Summary:")
            print(f"     Total logs: {len(audit_log)}")
            print(f"     Warnings: {warnings_count}")
            print(f"     Errors: {errors_count}")

            if warnings_count > 0:
                print(f"\n  ⚠️  Top Issues:")
                warning_logs = [log for log in audit_log if log['level'] == 'WARNING']
                for log in warning_logs[:5]:  # Show first 5 warnings
                    print(f"     - {log['message']}")

        # ✓ SAVE COMPLETE LOG (optional)
        if ENABLE_AUDIT_LOG and len(audit_log) > 0:
            audit_df = pd.DataFrame(audit_log)
            self.audit_log = audit_df  # Store for later analysis

        return results_df


    def get_trade_log(self) -> pd.DataFrame:
        if len(self.trades) == 0:
            return pd.DataFrame()
        return pd.DataFrame(self.trades)

    def get_regime_history(self) -> pd.DataFrame:
        """Return regime detection history as DataFrame"""
        if not self.regime_history:
            print("⚠️  Warning: Regime history is empty. Make sure REGIME_ENABLE = True")
            return pd.DataFrame()

        # Convert list of dicts to DataFrame
        regime_df = pd.DataFrame(self.regime_history)

        # Ensure Date column is datetime
        if 'Date' in regime_df.columns:
            regime_df['Date'] = pd.to_datetime(regime_df['Date'])

        # Sort by date
        regime_df = regime_df.sort_values('Date').reset_index(drop=True)

        print(f"✓ Regime history: {len(regime_df)} observations")
        return regime_df


    def print_last_rebalance_info(self, results_df: pd.DataFrame, data_dict: Dict):
        """Print information about last rebalance and current portfolio state"""

        if results_df.empty:
            print("No results to display")
            return

        last_rebalance = results_df[results_df['Rebalanced'] == True].index[-1] if any(results_df['Rebalanced']) else results_df.index[0]
        current_date = results_df.index[-1]
        last_date = results_df.index[-1]
        days_since = results_df.loc[last_date, 'Days_Since_Rebalance']

        # ✓ USAR EL VALOR GUARDADO
        rebal_freq = self.last_rebalance_frequency if self.last_rebalance_frequency else 21
        days_to_next = rebal_freq - days_since

        print("\n" + "=" * 80)
        print("LAST REBALANCE INFORMATION")
        print("=" * 80)

        print(f"\nLast Rebalance Date: {last_rebalance.date()}")
        print(f"Current Date: {current_date.date()}")
        print(f"Days Since Last Rebalance: {int(days_since)}")
        print(f"Days Until Next Rebalance: {int(days_to_next)}")
        print(f"Rebalance Frequency: Every {rebal_freq} trading days")

        # ✓ NEW: Show regime info
        if len(self.regime_history) > 0:
            last_regime = self.regime_history[-1]
            print(f"\n✓ Last Regime Detection (T-1):")
            print(f"  Regime: {last_regime['regime']}")
            print(f"  Exposure Multiplier: {last_regime['exposure_multiplier']:.2f}")
            print(f"  Volatility: {last_regime['volatility']:.2%}")
            print(f"  Trend Signal: {last_regime['trend_signal']}")

        long_positions = {k: v for k, v in self.last_positions.items() if v['side'] == 'long'}
        short_positions = {k: v for k, v in self.last_positions.items() if v['side'] == 'short'}

        print(f"\nCurrent Portfolio Composition:")
        print(f"  Long Positions: {len(long_positions)}")
        print(f"  Short Positions: {len(short_positions)}")

        if len(long_positions) > 0:
            print(f"\n  Top 10 Long Positions:")
            long_values = []
            for ticker, pos in long_positions.items():
                if ticker in data_dict and last_date in data_dict[ticker].index:
                    current_price = data_dict[ticker].loc[last_date, 'Adj_Close']
                    value = pos['shares'] * current_price
                    long_values.append((ticker, value, pos['shares']))

            long_values.sort(key=lambda x: abs(x[1]), reverse=True)
            for i, (ticker, value, shares) in enumerate(long_values[:10], 1):
                print(f"    {i:2d}. {ticker:6s}: ${value:8,.0f} ({shares:7,.1f} shares)")

        if len(short_positions) > 0:
            print(f"\n  Top 10 Short Positions:")
            short_values = []
            for ticker, pos in short_positions.items():
                if ticker in data_dict and last_date in data_dict[ticker].index:
                    current_price = data_dict[ticker].loc[last_date, 'Adj_Close']
                    value = pos['shares'] * current_price
                    short_values.append((ticker, value, pos['shares']))

            short_values.sort(key=lambda x: abs(x[1]), reverse=True)
            for i, (ticker, value, shares) in enumerate(short_values[:10], 1):
                print(f"    {i:2d}. {ticker:6s}: ${value:8,.0f} ({shares:7,.1f} shares)")

        print("\n" + "=" * 80)


# ============================================================================
# PERFORMANCE METRICS
# ============================================================================

class PerformanceAnalyzer:

    @staticmethod
    def calculate_sharpe(returns: pd.Series, rf_rate: float = 0.02) -> float:
        excess = returns - (rf_rate / 252)
        if excess.std() == 0:
            return 0.0
        return np.sqrt(252) * (excess.mean() / excess.std())

    @staticmethod
    def calculate_sortino(returns: pd.Series, rf_rate: float = 0.02) -> float:
        excess = returns - (rf_rate / 252)
        downside = excess[excess < 0]
        if len(downside) == 0 or downside.std() == 0:
            return 0.0
        return np.sqrt(252) * (excess.mean() / downside.std())

    @staticmethod
    def calculate_max_drawdown(cumulative_returns: pd.Series) -> float:
        cum_ret = 1 + cumulative_returns
        running_max = cum_ret.expanding().max()
        drawdown = (cum_ret - running_max) / running_max
        return drawdown.min()

    @staticmethod
    def calculate_cagr(returns: pd.Series) -> float:
        total_return = (1 + returns).prod()
        years = len(returns) / 252
        if years > 0:
            return (total_return ** (1 / years)) - 1
        return 0.0


# ============================================================================
# TURNOVER ANALYSIS MODULE
# ============================================================================

class TurnoverAnalyzer:
    """Calculate and penalize excessive turnover"""

    @staticmethod
    def calculate_turnover(trades: List[Dict], initial_capital: float) -> Dict:
        """
        Calculate portfolio turnover metrics

        Returns
        -------
        Dict with:
            - annual_turnover: Annualized turnover as % of capital
            - avg_holding_period: Average days per position
            - rebalance_frequency: Average days between rebalances
        """
        if len(trades) == 0:
            return {
                'annual_turnover': 0.0,
                'avg_holding_period': 0,
                'rebalance_frequency': 0,
                'total_trades': 0
            }

        # Convert to DataFrame
        trades_df = pd.DataFrame(trades)

        # Total notional traded
        total_traded = trades_df['Value'].abs().sum()

        # Calculate date range
        first_date = trades_df['Date'].min()
        last_date = trades_df['Date'].max()
        days_in_backtest = (last_date - first_date).days

        if days_in_backtest == 0:
            return {'annual_turnover': 0.0, 'avg_holding_period': 0,
                    'rebalance_frequency': 0, 'total_trades': len(trades)}

        # Annualized turnover
        turnover_rate = (total_traded / initial_capital) / (days_in_backtest / 365.25)

        # Rebalance frequency
        rebalance_dates = trades_df['Date'].unique()
        if len(rebalance_dates) > 1:
            rebalance_freq = days_in_backtest / (len(rebalance_dates) - 1)
        else:
            rebalance_freq = days_in_backtest

        return {
            'annual_turnover': turnover_rate,
            'avg_holding_period': rebalance_freq,
            'rebalance_frequency': rebalance_freq,
            'total_trades': len(trades)
        }

    @staticmethod
    def apply_turnover_penalty(sharpe: float, turnover_rate: float,
                               penalty_weight: float = 0.15) -> float:
        """
        Penalize Sharpe ratio for excessive turnover

        Parameters
        ----------
        sharpe : Raw Sharpe ratio
        turnover_rate : Annual turnover as decimal (e.g., 2.0 = 200%)
        penalty_weight : Sharpe reduction per 100% annual turnover

        Returns
        -------
        Adjusted Sharpe ratio

        Example
        -------
        sharpe = 2.0, turnover = 300%, penalty = 0.15
        adjusted_sharpe = 2.0 - (3.0 × 0.15) = 1.55
        """
        penalty = turnover_rate * penalty_weight
        adjusted_sharpe = sharpe - penalty

        return adjusted_sharpe



# ============================================================================
# ROBUSTNESS METRICS
# ============================================================================

class RobustnessMetrics:

    @staticmethod
    def deflated_sharpe_ratio(sharpe: float, n_trials: int,
                             skewness: float, kurtosis: float) -> float:
        if n_trials <= 1:
            return sharpe

        variance_adjustment = 1 - np.euler_gamma * np.log(n_trials) / np.sqrt(n_trials)
        moments_adjustment = skewness * sharpe + (kurtosis - 1) / 4 * sharpe**2
        dsr = sharpe * variance_adjustment - moments_adjustment
        return dsr

    @staticmethod
    def probabilistic_sharpe_ratio(sharpe_observed: float,
                                   sharpe_benchmark: float,
                                   n_observations: int,
                                   skewness: float,
                                   kurtosis: float) -> float:
        if n_observations <= 1:
            return 0.5

        sr_std = np.sqrt((1 + 0.5 * sharpe_observed**2 -
                         skewness * sharpe_observed +
                         (kurtosis - 3) / 4 * sharpe_observed**2) / n_observations)

        if sr_std == 0:
            return 1.0 if sharpe_observed > sharpe_benchmark else 0.0

        z = (sharpe_observed - sharpe_benchmark) / sr_std
        psr = stats.norm.cdf(z)

        return psr

    @staticmethod
    def walk_forward_efficiency(is_sharpe: float, oos_sharpe: float) -> float:
        if is_sharpe == 0:
            return 0.0
        return oos_sharpe / is_sharpe

    @staticmethod
    def hurst_exponent(returns: pd.Series) -> float:
        if len(returns) < 20:
            return 0.5

        lags = range(2, min(100, len(returns) // 2))
        tau = []

        for lag in lags:
            diffs = returns.rolling(lag).apply(lambda x: x[-1] - x[0], raw=True)
            tau.append(np.sqrt(diffs.dropna().std()))

        try:
            poly = np.polyfit(np.log(lags), np.log(tau), 1)
            return poly[0]
        except:
            return 0.5

    @staticmethod
    def conditional_drawdown_at_risk(returns: pd.Series, alpha: float = 0.05) -> float:
        cum_returns = (1 + returns).cumprod()
        running_max = cum_returns.expanding().max()
        drawdowns = (cum_returns - running_max) / running_max

        var_threshold = drawdowns.quantile(alpha)
        tail_drawdowns = drawdowns[drawdowns <= var_threshold]

        if len(tail_drawdowns) == 0:
            return 0.0

        return tail_drawdowns.mean()

    @staticmethod
    def parameter_stability_index(performances: List[float]) -> float:
        if len(performances) < 2:
            return 0.0

        performances = np.array(performances)
        mean_perf = np.mean(performances)

        if mean_perf == 0:
            return np.inf

        psi = np.std(performances) / abs(mean_perf)
        return psi

    @staticmethod
    def calculate_all_metrics(train_returns: pd.Series,
                             test_returns: pd.Series,
                             n_trials: int = 1,
                             parameter_performances: List[float] = None) -> Dict:

        train_sharpe = PerformanceAnalyzer.calculate_sharpe(train_returns)
        test_sharpe = PerformanceAnalyzer.calculate_sharpe(test_returns)

        test_skew = test_returns.skew()
        test_kurt = test_returns.kurtosis()

        dsr = RobustnessMetrics.deflated_sharpe_ratio(
            test_sharpe, n_trials, test_skew, test_kurt
        )

        psr = RobustnessMetrics.probabilistic_sharpe_ratio(
            test_sharpe, 0.0, len(test_returns), test_skew, test_kurt
        )

        wfe = RobustnessMetrics.walk_forward_efficiency(train_sharpe, test_sharpe)
        hurst = RobustnessMetrics.hurst_exponent(test_returns)
        cdar = RobustnessMetrics.conditional_drawdown_at_risk(test_returns)

        if parameter_performances and len(parameter_performances) > 1:
            psi = RobustnessMetrics.parameter_stability_index(parameter_performances)
        else:
            psi = 0.0

        mean_ic = 0.0

        metric_dict = {
            'dsr': dsr,
            'psr': psr,
            'wfe': wfe,
            'hurst': hurst,
            'cdar': cdar,
            'psi': psi,
            'mean_ic': mean_ic
        }

        normalized = {}
        normalized['dsr'] = max(0, min(1, (dsr + 2) / 4))
        normalized['psr'] = psr
        normalized['wfe'] = max(0, min(1, wfe / 2))
        hurst_dist = abs(hurst - 0.5)
        normalized['hurst'] = max(0, 1 - hurst_dist * 2)
        normalized['cdar'] = max(0, min(1, 1 + cdar * 2))
        normalized['psi'] = max(0, 1 - min(1, psi))
        normalized['ic'] = max(0, min(1, (mean_ic + 0.1) / 0.2))

        gri = np.mean(list(normalized.values()))
        metric_dict['gri'] = gri

        return metric_dict


# ============================================================================
# MONTE CARLO SIMULATION MODULE
# ============================================================================

class WalkForwardAnalyzer:
    """Walk-Forward Analysis with parameter optimization"""

    def __init__(self, config: WFAConfig):
        self.config = config
        self.results = []
        self.wfa_blocks = []
        self.backtest_engine = None

    def split_data_expanding(self, dates: pd.DatetimeIndex) -> List[Dict]:
        """Split data into expanding windows"""
        blocks = []

        min_dates = self.config.TRAIN_DAYS + self.config.VALIDATION_DAYS + self.config.TEST_DAYS

        if len(dates) < min_dates:
            print(f"Warning: Insufficient data. Need {min_dates}, have {len(dates)}")
            return blocks

        current_idx = 0
        block_num = 1

        while True:
            train_start_idx = current_idx
            train_end_idx = train_start_idx + self.config.TRAIN_DAYS

            val_end_idx = train_end_idx + self.config.VALIDATION_DAYS
            test_end_idx = val_end_idx + self.config.TEST_DAYS

            if test_end_idx > len(dates):
                break

            block = {
                'block_num': block_num,
                'train_start': dates[train_start_idx],
                'train_end': dates[train_end_idx - 1],
                'val_start': dates[train_end_idx],
                'val_end': dates[val_end_idx - 1],
                'test_start': dates[val_end_idx],
                'test_end': dates[test_end_idx - 1]
            }

            blocks.append(block)
            current_idx += self.config.TEST_DAYS
            block_num += 1

        return blocks

    def optimize_parameters(self, data_dict: Dict[str, pd.DataFrame],
                           train_start: pd.Timestamp, train_end: pd.Timestamp,
                           val_start: pd.Timestamp, val_end: pd.Timestamp) -> Tuple[Dict, Dict]:
        """
        Optimize parameters with TURNOVER PENALTY

        ✓ ENHANCED: Penalizes high turnover to promote stability
        """

        param_keys = list(self.config.OPTIMIZE_PARAMS.keys())
        param_values = list(self.config.OPTIMIZE_PARAMS.values())
        all_combinations = list(product(*param_values))

        print(f"    Testing {len(all_combinations)} parameter combinations...")

        best_val_metric = -np.inf
        best_params = None
        train_results_best = None

        for idx, combo in enumerate(all_combinations):
            if (idx + 1) % 10 == 0 or idx == len(all_combinations) - 1:
                print(f"      Progress: {idx + 1}/{len(all_combinations)}", end='\r')

            params = dict(zip(param_keys, combo))

            # ───────────────────────────────────────────────────────────────────
            # Run on TRAIN set
            # ───────────────────────────────────────────────────────────────────
            train_results = self._run_single_backtest(
                data_dict, train_start, train_end, params
            )

            if train_results.empty or len(train_results) < 10:
                continue

            train_returns = train_results['Daily_Return'].dropna()
            if len(train_returns) < 10:
                continue

            train_sharpe = PerformanceAnalyzer.calculate_sharpe(train_returns)

            # ✓ NEW: Calculate turnover on train set
            if self.config.ENABLE_TURNOVER_PENALTY:
                turnover_metrics = TurnoverAnalyzer.calculate_turnover(
                    self.backtest_engine.trades,
                    self.config.INITIAL_CAPITAL
                )

                # Apply penalty
                train_sharpe_adjusted = TurnoverAnalyzer.apply_turnover_penalty(
                    train_sharpe,
                    turnover_metrics['annual_turnover'],
                    self.config.TURNOVER_PENALTY_WEIGHT
                )
            else:
                train_sharpe_adjusted = train_sharpe
                turnover_metrics = {'annual_turnover': 0.0}

            # ───────────────────────────────────────────────────────────────────
            # Run on VALIDATION set
            # ───────────────────────────────────────────────────────────────────
            val_results = self._run_single_backtest(
                data_dict, val_start, val_end, params
            )

            if val_results.empty or len(val_results) < 5:
                continue

            val_returns = val_results['Daily_Return'].dropna()
            if len(val_returns) < 5:
                continue

            val_sharpe = PerformanceAnalyzer.calculate_sharpe(val_returns)

            # ✓ NEW: Also apply turnover penalty to validation
            if self.config.ENABLE_TURNOVER_PENALTY:
                val_turnover_metrics = TurnoverAnalyzer.calculate_turnover(
                    self.backtest_engine.trades,
                    self.config.INITIAL_CAPITAL
                )

                val_sharpe_adjusted = TurnoverAnalyzer.apply_turnover_penalty(
                    val_sharpe,
                    val_turnover_metrics['annual_turnover'],
                    self.config.TURNOVER_PENALTY_WEIGHT
                )
            else:
                val_sharpe_adjusted = val_sharpe

            # ───────────────────────────────────────────────────────────────────
            # Select best based on ADJUSTED validation Sharpe
            # ───────────────────────────────────────────────────────────────────
            if val_sharpe_adjusted > best_val_metric:
                best_val_metric = val_sharpe_adjusted
                best_params = params
                train_results_best = {
                    'sharpe': train_sharpe,
                    'sharpe_adjusted': train_sharpe_adjusted,
                    'turnover': turnover_metrics['annual_turnover'],
                    'results_df': train_results
                }

        if best_params is None:
            print("\n    ⚠️  WARNING: No valid parameter combination found")
            return {}, {}

        print(f"\n    ✓ Best validation Sharpe: {best_val_metric:.3f}")

        # ✓ NEW: Report turnover impact
        if self.config.ENABLE_TURNOVER_PENALTY and 'turnover' in train_results_best:
            print(f"      Raw Sharpe: {train_results_best['sharpe']:.3f}")
            print(f"      Adjusted Sharpe: {train_results_best['sharpe_adjusted']:.3f}")
            print(f"      Annual Turnover: {train_results_best['turnover']:.1f}x")
            penalty = train_results_best['turnover'] * self.config.TURNOVER_PENALTY_WEIGHT
            print(f"      Penalty Applied: -{penalty:.3f}")

        print(f"    Best parameters: {best_params}")

        return best_params, train_results_best

    def _run_single_backtest(self,
                            data_dict: Dict[str, pd.DataFrame],
                            start_date: pd.Timestamp,
                            end_date: pd.Timestamp,
                            params: Dict) -> pd.DataFrame:
        """
        Run a single backtest with given parameters
        Internal method used during optimization
        """

        # Extract parameters
        beta_window = params['BETA_WINDOW']
        beta_vol_window = params['BETA_VOL_WINDOW']
        long_pct = params['LONG_PERCENTILE']
        short_pct = params['SHORT_PERCENTILE']
        rebal_freq = params['REBALANCE_FREQUENCY']

        # Calculate betas
        beta_calc = BetaCalculator(self.config)
        beta_vols_df, _ = beta_calc.calculate_all_betas(
            data_dict, beta_window, beta_vol_window
        )


        # DEBUG:
        print(f"\n  🔍 Beta DataFrame Info:")
        print(f"     Shape: {beta_vols_df.shape}")
        print(f"     Dates: {beta_vols_df.index[0]} to {beta_vols_df.index[-1]}")
        print(f"     Tickers: {len(beta_vols_df.columns)}")
        print(f"     Non-null values per row (sample):")
        print(f"       {beta_vols_df.iloc[0].count()} at {beta_vols_df.index[0]}")
        print(f"       {beta_vols_df.iloc[-1].count()} at {beta_vols_df.index[-1]}")

        # Generate signals
        signal_gen = SignalGenerator(self.config)

        # Create new backtest engine for this run
        engine = BacktestEngine(self.config)

        # Run backtest
        results = engine.run_backtest(
            data_dict=data_dict,
            beta_vols_df=beta_vols_df,
            signal_generator=signal_gen,
            start_date=start_date,
            end_date=end_date,
            long_pct=long_pct,
            short_pct=short_pct,
            rebal_freq=rebal_freq
        )

        # Store engine for turnover calculation
        self.backtest_engine = engine

        return results

    def run_wfa(self, data_dict: Dict[str, pd.DataFrame]) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """Execute complete Walk-Forward Analysis"""

        print("\n" + "=" * 80)
        print("WALK-FORWARD ANALYSIS")
        print("=" * 80)

        market_data = data_dict[self.config.MARKET_BENCHMARK]
        backtest_start = pd.to_datetime(self.config.BACKTEST_START_DATE)
        backtest_end = pd.to_datetime(self.config.BACKTEST_END_DATE)

        all_dates = market_data.loc[
            (market_data.index >= backtest_start) &
            (market_data.index <= backtest_end)
        ].index

        print(f"Total dates available: {len(all_dates)}")
        print(f"Period: {all_dates[0].date()} to {all_dates[-1].date()}")

        wfa_blocks = self.split_data_expanding(all_dates)
        print(f"Created {len(wfa_blocks)} WFA blocks")

        if len(wfa_blocks) == 0:
            raise ValueError("No WFA blocks created - insufficient data")

        beta_calc = BetaCalculator(self.config)
        signal_gen = SignalGenerator(self.config)

        all_test_results = []
        block_summaries = []

        n_trials = len(list(product(*self.config.OPTIMIZE_PARAMS.values())))

        for block in wfa_blocks:
            print(f"\n{'─' * 80}")
            print(f"BLOCK {block['block_num']}")
            print(f"{'─' * 80}")

            print(f"\n  Optimizing parameters...")
            print(f"    Train: {block['train_start'].date()} to {block['train_end'].date()}")
            print(f"    Validate: {block['val_start'].date()} to {block['val_end'].date()}")

            best_params, train_results_best = self.optimize_parameters(
                data_dict,
                block['train_start'], block['train_end'],
                block['val_start'], block['val_end']
            )

            if not best_params:
                print(f"  ! Optimization failed for block {block['block_num']}")
                continue

            # Calculate betas with best params
            beta_vols_df, _ = beta_calc.calculate_all_betas(
                data_dict,
                best_params['BETA_WINDOW'],
                best_params['BETA_VOL_WINDOW']
            )

            # Run backtest on TEST set with best params
            backtest_engine = BacktestEngine(self.config)

            print(f"\n  Running on training set...")
            train_results = backtest_engine.run_backtest(
                data_dict, beta_vols_df, signal_gen,
                block['train_start'], block['train_end'],
                best_params['LONG_PERCENTILE'],
                best_params['SHORT_PERCENTILE'],
                best_params['REBALANCE_FREQUENCY']
            )

            print(f"  Running on test set...")
            test_results = backtest_engine.run_backtest(
                data_dict, beta_vols_df, signal_gen,
                block['test_start'], block['test_end'],
                best_params['LONG_PERCENTILE'],
                best_params['SHORT_PERCENTILE'],
                best_params['REBALANCE_FREQUENCY']
            )

            if len(test_results) < 5:
                print(f"  ! Insufficient test results for block {block['block_num']}")
                continue

            # Store last engine for regime history
            if block['block_num'] == len(wfa_blocks):
                self.backtest_engine = backtest_engine

            # Calculate metrics
            train_returns = train_results['Daily_Return'].dropna()
            test_returns = test_results['Daily_Return'].dropna()

            train_sharpe = PerformanceAnalyzer.calculate_sharpe(train_returns)
            test_sharpe = PerformanceAnalyzer.calculate_sharpe(test_returns)
            test_sortino = PerformanceAnalyzer.calculate_sortino(test_returns)
            test_cagr = PerformanceAnalyzer.calculate_cagr(test_returns)
            test_max_dd = PerformanceAnalyzer.calculate_max_drawdown(
                test_results['Cumulative_Return']
            )

            # Robustness metrics
            robustness = RobustnessMetrics.calculate_all_metrics(
                train_returns, test_returns, n_trials=n_trials
            )

            summary = {
                'Block': block['block_num'],
                'Test_Start': block['test_start'],
                'Test_End': block['test_end'],
                'Train_Sharpe': train_sharpe,
                'Val_Sharpe': train_results_best.get('sharpe_adjusted', 0.0),
                'Test_Sharpe': test_sharpe,
                'Test_Sortino': test_sortino,
                'Test_CAGR': test_cagr,
                'Test_Max_DD': test_max_dd,
                'WFE': robustness['wfe'],
                'DSR': robustness['dsr'],
                'PSR': robustness['psr'],
                'Hurst': robustness['hurst'],
                'CDaR': robustness['cdar'],
                'PSI': robustness['psi'],
                'GRI': robustness['gri'],
                **best_params  # Add parameters to summary
            }
            block_summaries.append(summary)

            test_results['Block'] = block['block_num']
            all_test_results.append(test_results)

            print(f"\n  ✓ Block {block['block_num']} Complete:")
            print(f"    Train Sharpe: {train_sharpe:.3f}")
            print(f"    Val Sharpe: {train_results_best.get('sharpe_adjusted', 0.0):.3f}")
            print(f"    Test Sharpe: {test_sharpe:.3f}")
            print(f"    Test CAGR: {test_cagr:.2%}")
            print(f"    WFE: {robustness['wfe']:.3f}")
            print(f"    GRI: {robustness['gri']:.3f}")

        if len(all_test_results) == 0:
            raise ValueError("No successful WFA blocks")

        combined_results = pd.concat(all_test_results, axis=0)
        summary_df = pd.DataFrame(block_summaries)

        print("\n" + "=" * 80)
        print("WFA SUMMARY")
        print("=" * 80)
        print(f"\nTotal Blocks: {len(summary_df)}")
        print(f"Average Test Sharpe: {summary_df['Test_Sharpe'].mean():.3f}")
        print(f"Average WFE: {summary_df['WFE'].mean():.3f}")
        print(f"Average GRI: {summary_df['GRI'].mean():.3f}")

        avg_gri = summary_df['GRI'].mean()
        if avg_gri > 0.65:
            print(f"  ✓ ROBUST: GRI = {avg_gri:.3f} > 0.65")
        else:
            print(f"  ! OVERFITTING RISK: GRI = {avg_gri:.3f} < 0.65")

        self.wfa_blocks = summary_df
        self.combined_results = combined_results

        return combined_results, summary_df



# ============================================================================
# VISUALIZATION
# ============================================================================

class WFAVisualizer:

    def __init__(self, config: WFAConfig):
        self.config = config

    def plot_wfa_equity_curve(self, combined_results: pd.DataFrame,
                          summary_df: pd.DataFrame,
                          benchmark_returns: pd.Series = None):
        """Plot equity curve with WFA blocks and benchmark"""

        if 'Cumulative_Return' not in combined_results.columns:
            print("⚠️  Missing 'Cumulative_Return' column")
            return

        fig, ax = plt.subplots(figsize=self.config.PLOT_FIGSIZE_WIDE, dpi=self.config.PLOT_DPI)

        # ═══════════════════════════════════════════════════════════════════════
        # SHADE WFA BLOCKS
        # ═══════════════════════════════════════════════════════════════════════

        # Define color palette for blocks (cycling)
        block_colors = ['lightblue', 'lightgreen', 'lightyellow', 'lightcoral',
                        'lavender', 'peachpuff', 'lightcyan', 'mistyrose',
                        'honeydew', 'lemonchiffon']

        test_blocks = []

        for _, row in summary_df.iterrows():
            block_num = int(row['Block'])
            test_start = pd.to_datetime(row['Test_Start'])
            test_end = pd.to_datetime(row['Test_End'])

            # Cycle through colors
            color = block_colors[(block_num - 1) % len(block_colors)]

            ax.axvspan(test_start, test_end, alpha=0.15, color=color, zorder=1)

            test_blocks.append({
                'block': block_num,
                'start': test_start,
                'end': test_end,
                'color': color
            })

        # ═══════════════════════════════════════════════════════════════════════
        # PLOT EQUITY CURVES
        # ═══════════════════════════════════════════════════════════════════════

        # Strategy
        ax.plot(combined_results.index,
                combined_results['Cumulative_Return'] * 100,
                color=self.config.PLOT_COLOR_STRATEGY,
                linewidth=self.config.PLOT_LINEWIDTH_MAIN,
                label='Strategy',
                zorder=10)

        # Benchmark
        if benchmark_returns is not None:
            benchmark_dates = combined_results.index.intersection(benchmark_returns.index)
            if len(benchmark_dates) > 0:
                bench_aligned = benchmark_returns.loc[benchmark_dates]
                bench_cumret = (1 + bench_aligned).cumprod() - 1

                ax.plot(bench_cumret.index,
                    bench_cumret * 100,
                    color=self.config.PLOT_COLOR_BENCHMARK,
                    linewidth=self.config.PLOT_LINEWIDTH_SECONDARY,
                    linestyle='--',
                    label='QQQ Benchmark',
                    alpha=0.8,
                    zorder=9)

        # Zero line
        ax.axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.5, zorder=2)

        # ═══════════════════════════════════════════════════════════════════════
        # ANNOTATIONS: Block numbers
        # ═══════════════════════════════════════════════════════════════════════

        for block_info in test_blocks:
            mid_date = block_info['start'] + (block_info['end'] - block_info['start']) / 2

            # Get y-position at mid-date
            if mid_date in combined_results.index:
                y_pos = combined_results.loc[mid_date, 'Cumulative_Return'] * 100
            else:
                # Find nearest date
                nearest_idx = combined_results.index.get_indexer([mid_date], method='nearest')[0]
                y_pos = combined_results.iloc[nearest_idx]['Cumulative_Return'] * 100

            # Add block number annotation
            ax.annotate(f'B{block_info["block"]}',
                    xy=(mid_date, y_pos),
                    xytext=(0, 20),
                    textcoords='offset points',
                    ha='center',
                    fontsize=8,
                    fontweight='bold',
                    color='darkblue',
                    alpha=0.7,
                    bbox=dict(boxstyle='round,pad=0.3',
                            facecolor=block_info['color'],
                            edgecolor='gray',
                            alpha=0.8))

        # ═══════════════════════════════════════════════════════════════════════
        # FORMATTING
        # ═══════════════════════════════════════════════════════════════════════

        ax.set_title('Walk-Forward Analysis - Cumulative Returns by Block',
                    fontsize=14, fontweight='bold', pad=15)
        ax.set_xlabel('Date', fontsize=11)
        ax.set_ylabel('Cumulative Return (%)', fontsize=11)
        ax.legend(loc='upper left', fontsize=10, framealpha=0.95)
        ax.grid(True, alpha=self.config.PLOT_GRID_ALPHA)

        # Format y-axis as percentage
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))

        # Add summary text box
        final_return = combined_results['Cumulative_Return'].iloc[-1] * 100
        max_dd = combined_results['Cumulative_Return'].cummin() - combined_results['Cumulative_Return']
        max_dd_pct = max_dd.max() * 100

        if 'Daily_Return' in combined_results.columns:
            daily_returns = combined_results['Daily_Return'].dropna()
            sharpe = PerformanceAnalyzer.calculate_sharpe(daily_returns)
        else:
            sharpe = 0.0

        textstr = f'Final Return: {final_return:.1f}%\n'
        textstr += f'Sharpe Ratio: {sharpe:.2f}\n'
        textstr += f'Max Drawdown: {max_dd_pct:.1f}%\n'
        textstr += f'Blocks: {len(summary_df)}'

        ax.text(0.02, 0.98, textstr,
            transform=ax.transAxes,
            fontsize=9,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

        plt.tight_layout()
        plt.show()

        print(f"✓ Plotted equity curve with {len(test_blocks)} blocks")


    def plot_regime_analysis(self, regime_df: pd.DataFrame, combined_results: pd.DataFrame):
        """Plot regime detection and exposure adjustments"""

        if regime_df is None or regime_df.empty:
            print("⚠️  No regime data available for plotting.")
            return

        required_cols = ["Date", "regime", "exposure_multiplier", "volatility"]
        missing = [c for c in required_cols if c not in regime_df.columns]
        if missing:
            print(f"⚠️  Missing regime columns: {missing}")
            return

        regime_df = regime_df.copy()
        regime_df["Date"] = pd.to_datetime(regime_df["Date"])
        regime_df = regime_df.sort_values("Date").set_index("Date")
        regime_df = regime_df.ffill().bfill()

        if len(regime_df) < 2:
            print(f"⚠️  Insufficient regime data: {len(regime_df)} observations")
            return

        if "Daily_Return" in combined_results.columns:
            returns_aligned = combined_results["Daily_Return"].reindex(regime_df.index).fillna(0)
            cum_ret_regime = (1 + returns_aligned).cumprod() - 1
        else:
            print("⚠️  Missing 'Daily_Return' column")
            return

        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 12), dpi=self.config.PLOT_DPI)

        # Panel 1: Exposure
        ax1.plot(regime_df.index, regime_df["exposure_multiplier"],
                color="navy", linewidth=2.5)
        ax1.fill_between(regime_df.index, 0, regime_df["exposure_multiplier"],
                        color="skyblue", alpha=0.3)
        ax1.axhline(1.0, color="green", linestyle="--", linewidth=1.5, alpha=0.7)
        ax1.set_title("Regime-Based Exposure Adjustments", fontsize=13, fontweight="bold")
        ax1.set_ylabel("Exposure Multiplier", fontsize=11)
        ax1.grid(alpha=0.3)
        ax1.set_ylim(0, 1.1)

        # Panel 2: Volatility
        ax2.plot(regime_df.index, regime_df["volatility"] * 100,
                color="darkred", linewidth=2.5)
        ax2.axhline(self.config.REGIME_VOL_THRESHOLD * 100,
                color="red", linestyle="--", linewidth=1.5, alpha=0.7)
        ax2.set_title("Market Volatility (Annualized)", fontsize=13, fontweight="bold")
        ax2.set_ylabel("Volatility (%)", fontsize=11)
        ax2.grid(alpha=0.3)

        # Panel 3: Returns by regime
        ax3.plot(cum_ret_regime.index, cum_ret_regime * 100,
                color=self.config.PLOT_COLOR_STRATEGY, linewidth=2.5)

        regime_colors = {
            "high_vol_down": "darkred",
            "high_vol": "orange",
            "downtrend": "khaki",
            "uptrend": "lightgreen",
            "neutral": "lightgray"
        }

        for i in range(len(regime_df) - 1):
            start_date = regime_df.index[i]
            end_date = regime_df.index[i + 1]
            regime = regime_df["regime"].iloc[i]
            color = regime_colors.get(regime, "lightgray")
            ax3.axvspan(start_date, end_date, color=color, alpha=0.2)

        ax3.set_title("Cumulative Returns by Regime", fontsize=13, fontweight="bold")
        ax3.set_xlabel("Date", fontsize=11)
        ax3.set_ylabel("Cumulative Return (%)", fontsize=11)
        ax3.axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.6)
        ax3.grid(alpha=0.3)

        plt.tight_layout()
        plt.show()


    def plot_financing_costs(self, combined_results: pd.DataFrame):
        """✓ NEW: Plot breakdown of financing costs"""

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), dpi=self.config.PLOT_DPI)

        # Cumulative costs over time
        cum_borrow = combined_results['Borrow_Cost'].cumsum()
        cum_margin = combined_results['Margin_Interest'].cumsum()
        cum_divs = combined_results['Short_Dividends'].cumsum()
        cum_total = combined_results['Total_Financing'].cumsum()

        ax1.plot(cum_borrow.index, cum_borrow.values,
                label='Stock Borrow Cost', linewidth=2, color='blue')
        ax1.plot(cum_margin.index, cum_margin.values,
                label='Margin Interest', linewidth=2, color='red')
        ax1.plot(cum_divs.index, cum_divs.values,
                label='Short Dividends', linewidth=2, color='orange')
        ax1.plot(cum_total.index, cum_total.values,
                label='Total Financing', linewidth=2.5, color='black', linestyle='--')

        ax1.set_title('Cumulative Financing Costs', fontsize=13, fontweight='bold')
        ax1.set_ylabel('Cumulative Cost ($)', fontsize=11)
        ax1.legend(loc='best', fontsize=10)
        ax1.grid(True, alpha=0.3)

        # Daily financing costs
        ax2.plot(combined_results.index, combined_results['Total_Financing'],
                label='Daily Total Financing', linewidth=1.5, color='darkred', alpha=0.7)

        ax2.set_title('Daily Financing Costs', fontsize=13, fontweight='bold')
        ax2.set_xlabel('Date', fontsize=11)
        ax2.set_ylabel('Daily Cost ($)', fontsize=11)
        ax2.legend(loc='best', fontsize=10)
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # Print summary
        print("\n" + "=" * 80)
        print("FINANCING COST SUMMARY")
        print("=" * 80)
        print(f"\nTotal Borrow Cost: ${cum_borrow.iloc[-1]:,.2f}")
        print(f"Total Margin Interest: ${cum_margin.iloc[-1]:,.2f}")
        print(f"Total Short Dividends: ${cum_divs.iloc[-1]:,.2f}")
        print(f"Total Financing Cost: ${cum_total.iloc[-1]:,.2f}")
        print(f"\nAs % of Initial Capital: {cum_total.iloc[-1] / self.config.INITIAL_CAPITAL * 100:.2f}%")

    def plot_rebalancing_events(self, combined_results: pd.DataFrame):

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), dpi=self.config.PLOT_DPI)

        rebalance_dates = combined_results[combined_results['Rebalanced'] == True].index

        ax1.plot(combined_results.index, combined_results['Portfolio_Value'],
                color=self.config.PLOT_COLOR_STRATEGY, linewidth=2)

        rebal_values = combined_results.loc[rebalance_dates, 'Portfolio_Value']
        ax1.scatter(rebalance_dates, rebal_values,
                   color='red', s=50, alpha=0.6, zorder=5,
                   label=f'Rebalance Events (n={len(rebalance_dates)})')

        ax1.axhline(self.config.INITIAL_CAPITAL, color='black',
                   linestyle='--', linewidth=1, alpha=0.5,
                   label='Initial Capital')

        ax1.set_title('Portfolio Value with Rebalancing Events',
                     fontsize=13, fontweight='bold')
        ax1.set_ylabel('Portfolio Value ($)', fontsize=11)
        ax1.legend(loc='best', fontsize=10)
        ax1.grid(True, alpha=0.3)

        ax2.plot(combined_results.index, combined_results['Num_Long'],
                label='Long Positions',
                color=self.config.PLOT_COLOR_LONG, linewidth=2)
        ax2.plot(combined_results.index, combined_results['Num_Short'],
                label='Short Positions',
                color=self.config.PLOT_COLOR_SHORT, linewidth=2)

        for date in rebalance_dates:
            ax2.axvline(date, color='red', alpha=0.2, linewidth=1, linestyle='--')

        ax2.set_title('Position Count Over Time',
                     fontsize=13, fontweight='bold')
        ax2.set_xlabel('Date', fontsize=11)
        ax2.set_ylabel('Number of Positions', fontsize=11)
        ax2.legend(loc='best', fontsize=10)
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

    def plot_block_performance(self, summary_df: pd.DataFrame):

        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 10), dpi=self.config.PLOT_DPI)

        blocks = summary_df['Block'].values

        width = 0.25
        x = np.arange(len(blocks))

        ax1.bar(x - width, summary_df['Train_Sharpe'], width,
               label='Train', color=self.config.PLOT_COLOR_TRAIN, alpha=0.8)
        ax1.bar(x, summary_df['Val_Sharpe'], width,
               label='Validation', color=self.config.PLOT_COLOR_VALIDATION, alpha=0.8)
        ax1.bar(x + width, summary_df['Test_Sharpe'], width,
               label='Test', color=self.config.PLOT_COLOR_TEST, alpha=0.8)

        ax1.set_title('Sharpe Ratio by Block', fontsize=12, fontweight='bold')
        ax1.set_xlabel('Block', fontsize=10)
        ax1.set_ylabel('Sharpe Ratio', fontsize=10)
        ax1.set_xticks(x)
        ax1.set_xticklabels(blocks)
        ax1.legend(fontsize=9)
        ax1.grid(True, alpha=0.3, axis='y')
        ax1.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)

        ax2.bar(blocks, summary_df['WFE'], color=self.config.PLOT_COLOR_STRATEGY, alpha=0.8)
        ax2.axhline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold (0.5)')
        ax2.set_title('Walk-Forward Efficiency (WFE)', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Block', fontsize=10)
        ax2.set_ylabel('WFE', fontsize=10)
        ax2.legend(fontsize=9)
        ax2.grid(True, alpha=0.3, axis='y')

        ax3.plot(blocks, summary_df['DSR'], marker='o', label='DSR', linewidth=2)
        ax3.plot(blocks, summary_df['PSR'], marker='s', label='PSR', linewidth=2)
        ax3.plot(blocks, summary_df['GRI'], marker='^', label='GRI', linewidth=2.5,
                color='darkgreen')
        ax3.axhline(0.65, color='red', linestyle='--', linewidth=1.5,
                   alpha=0.7, label='GRI Threshold')
        ax3.set_title('Robustness Metrics by Block', fontsize=12, fontweight='bold')
        ax3.set_xlabel('Block', fontsize=10)
        ax3.set_ylabel('Metric Value', fontsize=10)
        ax3.legend(fontsize=9)
        ax3.grid(True, alpha=0.3)

        ax4.bar(blocks, summary_df['Test_Max_DD'] * 100,
               color='darkred', alpha=0.7)
        ax4.set_title('Maximum Drawdown by Block', fontsize=12, fontweight='bold')
        ax4.set_xlabel('Block', fontsize=10)
        ax4.set_ylabel('Max Drawdown (%)', fontsize=10)
        ax4.grid(True, alpha=0.3, axis='y')

        plt.tight_layout()
        plt.show()

    def plot_robustness_summary(self, summary_df: pd.DataFrame):

        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 10), dpi=self.config.PLOT_DPI)

        ax1.hist(summary_df['GRI'], bins=10, color=self.config.PLOT_COLOR_STRATEGY,
                alpha=0.7, edgecolor='black')
        ax1.axvline(0.65, color='red', linestyle='--', linewidth=2,
                   label='Threshold (0.65)')
        ax1.axvline(summary_df['GRI'].mean(), color='green', linestyle='-',
                   linewidth=2, label=f'Mean ({summary_df["GRI"].mean():.3f})')
        ax1.set_title('Global Robustness Index Distribution', fontsize=12, fontweight='bold')
        ax1.set_xlabel('GRI', fontsize=10)
        ax1.set_ylabel('Frequency', fontsize=10)
        ax1.legend(fontsize=9)
        ax1.grid(True, alpha=0.3)

        ax2.scatter(summary_df['Block'], summary_df['Hurst'],
                   s=100, alpha=0.6, color=self.config.PLOT_COLOR_STRATEGY)
        ax2.axhline(0.5, color='red', linestyle='--', linewidth=2,
                   label='Random Walk (0.5)')
        ax2.fill_between(summary_df['Block'], 0.45, 0.55,
                        alpha=0.2, color='green', label='Acceptable Range')
        ax2.set_title('Hurst Exponent by Block', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Block', fontsize=10)
        ax2.set_ylabel('Hurst Exponent', fontsize=10)
        ax2.set_ylim(0, 1)
        ax2.legend(fontsize=9)
        ax2.grid(True, alpha=0.3)

        ax3.bar(summary_df['Block'], summary_df['PSI'],
               color=self.config.PLOT_COLOR_VALIDATION, alpha=0.7)
        ax3.set_title('Parameter Stability Index (PSI)', fontsize=12, fontweight='bold')
        ax3.set_xlabel('Block', fontsize=10)
        ax3.set_ylabel('PSI (Lower = More Stable)', fontsize=10)
        ax3.grid(True, alpha=0.3, axis='y')

        ax4.bar(summary_df['Block'], summary_df['CDaR'] * 100,
               color='darkred', alpha=0.7)
        ax4.set_title('Conditional Drawdown at Risk (95%)', fontsize=12, fontweight='bold')
        ax4.set_xlabel('Block', fontsize=10)
        ax4.set_ylabel('CDaR (%)', fontsize=10)
        ax4.grid(True, alpha=0.3, axis='y')

        plt.tight_layout()
        plt.show()

# ============================================================================
# POSITION MONITOR
# ============================================================================

class PositionMonitor:
    """Utility module to inspect current positions and rebalance schedule."""

    def __init__(self, combined_results: pd.DataFrame, config):
        self.results = combined_results.copy()
        self.config = config
        self._validate()

    def _validate(self):
        if "Date" not in self.results.columns:
            raise ValueError("combined_results must contain a 'Date' column")
        self.results["Date"] = pd.to_datetime(self.results["Date"])
        self.results = self.results.sort_values("Date")

    def get_last_rebalance_date(self):
        """Return the most recent rebalance date based on config.REBALANCE_DAYS"""
        if not hasattr(self.config, "REBALANCE_DAYS"):
            raise AttributeError("Config must include REBALANCE_DAYS")
        last_date = self.results["Date"].iloc[-1]
        total_days = len(self.results)
        rebalance_freq = getattr(self.config, "REBALANCE_DAYS", 20)
        last_reb_idx = (total_days // rebalance_freq) * rebalance_freq - 1
        last_reb_idx = max(0, min(last_reb_idx, total_days - 1))
        return self.results["Date"].iloc[last_reb_idx]

    def get_next_rebalance_date(self):
        """Estimate the next rebalance date after the latest run"""
        last_reb = self.get_last_rebalance_date()
        next_reb = last_reb + timedelta(days=self.config.REBALANCE_DAYS)
        end_date = self.results["Date"].max()
        return min(next_reb, end_date)

    def get_latest_positions(self):
        """Extract the last recorded portfolio composition"""
        cols = [c for c in self.results.columns if "Position" in c or "Weight" in c]
        if not cols:
            print("⚠ No position data columns found in combined_results.")
            return pd.DataFrame()
        last_row = self.results.iloc[-1:][["Date"] + cols]
        return last_row.T.rename(columns={last_row.index[-1]: "Latest_Values"})

    def summary(self):
        """Display a complete snapshot of rebalance and position status"""
        last_reb = self.get_last_rebalance_date()
        next_reb = self.get_next_rebalance_date()
        last_positions = self.get_latest_positions()

        print("\n=== POSITION MONITOR ===")
        print(f"Last Rebalance Date:   {last_reb.date()}")
        print(f"Next Scheduled Rebal.: {next_reb.date()}")
        print(f"Strategy Horizon End:  {self.results['Date'].max().date()}")
        print("\nCurrent Portfolio Snapshot:")
        if not last_positions.empty:
            print(last_positions)
        else:
            print("   No position columns detected.\n")


# ============================================================================
# UTILITY FUNCTIONS (BEFORE main)
# ============================================================================



def print_parameter_stability_report(summary_df: pd.DataFrame):
    """
    Print parameter variation analysis

    ✓ ADD THIS FUNCTION HERE (before main)
    """

    print("\n" + "="*80)
    print("PARAMETER STABILITY ANALYSIS")
    print("="*80)

    # Extract parameter columns (if they exist)
    param_cols = ['BETA_WINDOW', 'BETA_VOL_WINDOW', 'LONG_PERCENTILE',
                  'SHORT_PERCENTILE', 'REBALANCE_FREQUENCY']

    available_params = [col for col in param_cols if col in summary_df.columns]

    if len(available_params) == 0:
        print("⚠️  No parameter data available in summary_df")
        return

    print("\nParameter Distribution Across Blocks:")
    print("-" * 80)

    for param in available_params:
        values = summary_df[param].dropna()
        if len(values) > 0:
            print(f"\n{param}:")
            print(f"  Unique values: {values.nunique()}")
            print(f"  Most common: {values.mode().iloc[0] if len(values.mode()) > 0 else 'N/A'}")
            print(f"  Distribution: {values.value_counts().to_dict()}")

    # Calculate parameter stability (lower is better)
    param_variance = 0
    for param in available_params:
        if summary_df[param].dtype in [np.float64, np.int64]:
            param_variance += summary_df[param].var()

    print(f"\n📊 Parameter Variance Score: {param_variance:.2f}")
    print("   (Lower = more stable parameters across blocks)")

    # NEW: Turnover analysis if available
    if 'Annual_Turnover' in summary_df.columns:
        avg_turnover = summary_df['Annual_Turnover'].mean()
        print(f"\n📈 Average Annual Turnover: {avg_turnover:.1f}x")
        print(f"   Max: {summary_df['Annual_Turnover'].max():.1f}x")
        print(f"   Min: {summary_df['Annual_Turnover'].min():.1f}x")

# ============================================================================
# UTILITY FUNCTIONS (BEFORE main)
# ============================================================================



def print_parameter_stability_report(summary_df: pd.DataFrame):
    """
    Print parameter variation analysis

    ✓ ADD THIS FUNCTION HERE (before main)
    """

    print("\n" + "="*80)
    print("PARAMETER STABILITY ANALYSIS")
    print("="*80)

    # Extract parameter columns (if they exist)
    param_cols = ['BETA_WINDOW', 'BETA_VOL_WINDOW', 'LONG_PERCENTILE',
                  'SHORT_PERCENTILE', 'REBALANCE_FREQUENCY']

    available_params = [col for col in param_cols if col in summary_df.columns]

    if len(available_params) == 0:
        print("⚠️  No parameter data available in summary_df")
        return

    print("\nParameter Distribution Across Blocks:")
    print("-" * 80)

    for param in available_params:
        values = summary_df[param].dropna()
        if len(values) > 0:
            print(f"\n{param}:")
            print(f"  Unique values: {values.nunique()}")
            print(f"  Most common: {values.mode().iloc[0] if len(values.mode()) > 0 else 'N/A'}")
            print(f"  Distribution: {values.value_counts().to_dict()}")

    # Calculate parameter stability (lower is better)
    param_variance = 0
    for param in available_params:
        if summary_df[param].dtype in [np.float64, np.int64]:
            param_variance += summary_df[param].var()

    print(f"\n📊 Parameter Variance Score: {param_variance:.2f}")
    print("   (Lower = more stable parameters across blocks)")

    # NEW: Turnover analysis if available
    if 'Annual_Turnover' in summary_df.columns:
        avg_turnover = summary_df['Annual_Turnover'].mean()
        print(f"\n📈 Average Annual Turnover: {avg_turnover:.1f}x")
        print(f"   Max: {summary_df['Annual_Turnover'].max():.1f}x")
        print(f"   Min: {summary_df['Annual_Turnover'].min():.1f}x")




def main():
    """Execute Walk-Forward Analysis with Regime Detection"""

    print("\n" + "=" * 80)
    print("BETA INSTABILITY STRATEGY - ENHANCED WALK-FORWARD ANALYSIS")
    print("✓ Regime Detection + Complete Financing + Explicit T-1/T Flow")
    print("=" * 80)

    config = WFAConfig()

    print(f"\nConfiguration:")
    print(f"  Universe: {config.UNIVERSE}")
    print(f"  Benchmark: {config.BENCHMARK_TICKER}")
    print(f"  Period: {config.BACKTEST_START_DATE} to {config.BACKTEST_END_DATE}")
    print(f"  WFA Structure: {config.TRAIN_DAYS}d train + {config.VALIDATION_DAYS}d val + {config.TEST_DAYS}d test")
    print(f"  Compounding Mode: {config.COMPOUNDING_MODE} (Research mode)")
    print(f"  Initial Capital: ${config.INITIAL_CAPITAL:,.0f}")
    print(f"  Leverage: {config.LEVERAGE}x")
    print(f"\n✓ Regime Detection:")
    print(f"  Enabled: {config.REGIME_ENABLE}")
    print(f"  Volatility Threshold: {config.REGIME_VOL_THRESHOLD:.1%}")
    print(f"  Vol Window: {config.REGIME_VOL_WINDOW}d | Trend Window: {config.REGIME_TREND_WINDOW}d")
    print(f"\n✓ Complete Financing:")
    print(f"  Stock Borrow: {config.BORROW_COST_BPS} bps/year")
    print(f"  Margin Interest: {config.MARGIN_INTEREST_RATE:.2%}/year")
    print(f"  Short Dividends: {config.SHORT_DIVIDEND_RATE:.2%}/year")
    print(f"  Slippage: {config.SLIPPAGE_BPS} bps (base) | {config.SLIPPAGE_BPS_REBALANCE} bps (rebalance)")

    fetcher = DataFetcher()
    data_dict = fetcher.fetch_universe_data(config)

    wfa = WalkForwardAnalyzer(config)
    combined_results, summary_df = wfa.run_wfa(data_dict)

    if config.BENCHMARK_TICKER in data_dict:
        bench_data = data_dict[config.BENCHMARK_TICKER]
        bench_dates = combined_results.index.intersection(bench_data.index)
        bench_prices = bench_data.loc[bench_dates, 'Adj_Close']
        benchmark_returns = bench_prices.pct_change()
    else:
        benchmark_returns = None

    # Get regime history from last block
    if wfa.backtest_engine is not None:
        wfa.backtest_engine.print_last_rebalance_info(combined_results, data_dict)
        regime_df = wfa.backtest_engine.get_regime_history()
    else:
        regime_df = pd.DataFrame()

    print("\n" + "=" * 80)
    print("DETAILED METRICS BY BLOCK")
    print("=" * 80)

    display_cols = ['Block', 'Test_Start', 'Test_End', 'Train_Sharpe', 'Val_Sharpe',
                    'Test_Sharpe', 'Test_CAGR', 'Test_Max_DD', 'WFE', 'GRI']
    print("\n" + summary_df[display_cols].to_string(index=False))

    # ═══════════════════════════════════════════════════════════════════════
    # AGGREGATE STATISTICS
    # ═══════════════════════════════════════════════════════════════════════

    print("\n" + "=" * 80)
    print("AGGREGATE STATISTICS")
    print("=" * 80)

    print(f"\nPerformance Metrics:")
    print(f"  Average Test Sharpe: {summary_df['Test_Sharpe'].mean():.3f} ± {summary_df['Test_Sharpe'].std():.3f}")
    print(f"  Average Test CAGR: {summary_df['Test_CAGR'].mean():.2%}")
    print(f"  Average Max Drawdown: {summary_df['Test_Max_DD'].mean():.2%}")

    print(f"\nRobustness Metrics:")
    print(f"  Average WFE: {summary_df['WFE'].mean():.3f} (>0.5 is good)")
    print(f"  Average DSR: {summary_df['DSR'].mean():.3f}")
    print(f"  Average PSR: {summary_df['PSR'].mean():.3f}")
    print(f"  Average GRI: {summary_df['GRI'].mean():.3f} (>0.65 is robust)")
    print(f"  Average Hurst: {summary_df['Hurst'].mean():.3f} (0.5 = random walk)")
    print(f"  Average CDaR: {summary_df['CDaR'].mean():.2%}")
    print(f"  Average PSI: {summary_df['PSI'].mean():.3f} (lower is better)")

    # ✓ CALL THE NEW FUNCTION HERE
    print_parameter_stability_report(summary_df)

    # Continue with robustness assessment
    print(f"\n{'─' * 80}")
    print("OVERALL ROBUSTNESS ASSESSMENT")
    print(f"{'─' * 80}")

    avg_gri = summary_df['GRI'].mean()
    avg_wfe = summary_df['WFE'].mean()
    avg_psr = summary_df['PSR'].mean()

    criteria = {
        'GRI > 0.65': avg_gri > 0.65,
        'WFE > 0.50': avg_wfe > 0.50,
        'PSR > 0.70': avg_psr > 0.70
    }

    passed = sum(criteria.values())
    total = len(criteria)

    print(f"\nRobustness Criteria ({passed}/{total} passed):")
    for criterion, result in criteria.items():
        status = "✓ PASS" if result else "✗ FAIL"
        print(f"  {criterion}: {status}")

    if passed >= 2:
        print(f"\n✓ CONCLUSION: Strategy shows STATISTICAL ROBUSTNESS")
    else:
        print(f"\n! CONCLUSION: Strategy shows OVERFITTING RISK")

    # ═══════════════════════════════════════════════════════════════════════
    # GENERATING VISUALIZATIONS
    # ═══════════════════════════════════════════════════════════════════════

    print("\n" + "=" * 80)
    print("GENERATING VISUALIZATIONS")
    print("=" * 80)

    visualizer = WFAVisualizer(config)

    print("\n1. Equity Curve with WFA Blocks...")
    visualizer.plot_wfa_equity_curve(combined_results, summary_df, benchmark_returns)

    print("\n2. Regime Analysis...")
    visualizer.plot_regime_analysis(regime_df, combined_results)

    print("\n3. Financing Costs Breakdown...")
    visualizer.plot_financing_costs(combined_results)

    print("\n4. Rebalancing Events Analysis...")
    visualizer.plot_rebalancing_events(combined_results)

    print("\n5. Block Performance Comparison...")
    visualizer.plot_block_performance(summary_df)

    print("\n6. Robustness Metrics Summary...")
    visualizer.plot_robustness_summary(summary_df)

    # ═══════════════════════════════════════════════════════════════════════
    # ✓ AUDIT LOG ANALYSIS (NEW SECTION)
    # ═══════════════════════════════════════════════════════════════════════

    print("\n" + "="*80)
    print("8. AUDIT LOG ANALYSIS")
    print("="*80)

    # Check if audit log exists
    if hasattr(wfa.backtest_engine, 'audit_log'):
        audit_log = wfa.backtest_engine.audit_log

        # Summary statistics
        total_logs = len(audit_log)
        warnings = audit_log[audit_log['level'] == 'WARNING']
        errors = audit_log[audit_log['level'] == 'ERROR']

        print(f"\n📋 Complete Audit Summary:")
        print(f"  Total events logged: {total_logs}")
        print(f"  Warnings: {len(warnings)}")
        print(f"  Errors: {len(errors)}")

        # Show all warnings (if any)
        if len(warnings) > 0:
            print(f"\n⚠️  ALL WARNINGS DETECTED:")
            print("-" * 80)
            for idx, row in warnings.iterrows():
                print(f"{idx + 1}. {row['message']}")

            # Save to CSV for detailed analysis
            warnings.to_csv('audit_warnings.csv', index=False)
            print(f"\n✓ Warnings saved to: audit_warnings.csv")

        # Show all errors (if any)
        if len(errors) > 0:
            print(f"\n❌ ALL ERRORS DETECTED:")
            print("-" * 80)
            for idx, row in errors.iterrows():
                print(f"{idx + 1}. {row['message']}")

            # Save to CSV
            errors.to_csv('audit_errors.csv', index=False)
            print(f"\n✓ Errors saved to: audit_errors.csv")

        # Identify extreme return events
        extreme_returns = warnings[warnings['message'].str.contains('EXTREME RETURN', na=False)]
        if len(extreme_returns) > 0:
            print(f"\n🔥 EXTREME RETURN EVENTS: {len(extreme_returns)}")
            print("-" * 80)
            for idx, row in extreme_returns.iterrows():
                print(f"\n{row['message']}\n")

        # Identify position caps applied
        position_caps = warnings[warnings['message'].str.contains('CAPPED', na=False)]
        if len(position_caps) > 0:
            print(f"\n✂️  POSITION CAPS APPLIED: {len(position_caps)}")
            print("-" * 80)
            for idx, row in position_caps.iterrows():
                print(f"  - {row['message']}")

        # Identify extreme gaps
        extreme_gaps = warnings[warnings['message'].str.contains('EXTREME GAP', na=False)]
        if len(extreme_gaps) > 0:
            print(f"\n📊 EXTREME GAPS DETECTED: {len(extreme_gaps)}")
            print("-" * 80)
            for idx, row in extreme_gaps.iterrows():
                print(f"  - {row['message']}")
    else:
        print("\n⚠️  No audit log available.")
        print("   Make sure ENABLE_AUDIT_LOG = True in BacktestEngine.run_backtest()")

    # ═══════════════════════════════════════════════════════════════════════
    # FINAL SUMMARY
    # ═══════════════════════════════════════════════════════════════════════

    print("\n" + "="*80)
    print("✅ ENHANCED WALK-FORWARD ANALYSIS COMPLETE")
    print("="*80)

    return combined_results, summary_df, regime_df, config


# ============================================================================
# RUN THE ANALYSIS
# ============================================================================

if __name__ == "__main__":
    combined_results, summary_df, regime_df, config = main()
